In [ ]:
# Import mosaic libraries
import missionbio.mosaic as ms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import plotly.express as px
from scipy.stats import fisher_exact
from scipy.stats import mannwhitneyu

### Gene-level burden of coding somatic variants

In [ ]:
def annotate_spliceai_effect(df: pd.DataFrame) -> pd.DataFrame:
    cols = ['SpliceAI_pred_DS_AG','SpliceAI_pred_DS_AL','SpliceAI_pred_DS_DG','SpliceAI_pred_DS_DL']
    labels = ['acceptor_gain', 'acceptor_loss','donor_gain','donor_loss']
    ds_matrix = (df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy())
    max_idx = ds_matrix.argmax(axis=1)
    max_score = ds_matrix.max(axis=1)
    # Map severity
    severity = np.select(
        [max_score >= 0.8,max_score >= 0.5,max_score >= 0.2],
        ['high','moderate','low'],
        default='none'
    )
    # Map effect
    effect = [labels[i] for i in max_idx]
    effect = np.where(max_score == 0, 'none', effect)
    df['spliceai_severity'] = severity
    df['spliceai_effect'] = effect

    return df

def categorize_hc_lof(row) -> str:
    lof_flag = str(row.get("LoF", "")).upper()
    spliceai_sev = str(row.get("spliceai_severity", "")).lower()
    if lof_flag in {"HC", "OS"} or spliceai_sev == "high":
        return "Yes"
    return np.nan

output_dir = "/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data"

LOF = {
    "stop_gained", "frameshift_variant", "stop_gained&frameshift_variant",
    "splice_acceptor_variant", "splice_donor_variant",
    "stop_gained&splice_region_variant", "frameshift_variant&splice_region_variant"
}

indel_terms = [
    "frameshift_variant",
    "frameshift_variant&splice_region_variant",
    "stop_gained&frameshift_variant",
    "stop_gained&inframe_deletion",
    "inframe_deletion",
    "inframe_insertion",
    "frameshift_variant&stop_lost",
    "frameshift_variant&start_lost",
]

non_coding_terms = [
    "intron_variant",
    "splice_polypyrimidine_tract_variant&intron_variant",
    "splice_region_variant&splice_polypyrimidine_tract_variant&intron_variant",
    "splice_donor_region_variant&intron_variant",
    "splice_region_variant&5_prime_UTR_variant",
    "splice_donor_5th_base_variant&intron_variant",
    "splice_region_variant&intron_variant",
    "3_prime_UTR_variant",
    "5_prime_UTR_variant",
    # "splice_acceptor_variant", "splice_donor_variant"
]

splice_site_terms = ["splice_acceptor_variant", "splice_donor_variant"]
missense_terms = ["missense_variant", "missense_variant&splice_region_variant"]
nonsense_terms = ["stop_gained", "stop_gained&splice_region_variant", "stop_gained&frameshift_variant", "stop_gained&inframe_deletion"]
frameshift_terms = ["frameshift_variant", "frameshift_variant&splice_region_variant", "frameshift_variant&stop_lost", "frameshift_variant&start_lost"]
synonymous_terms = ['synonymous_variant', 'stop_retained_variant', 'splice_region_variant&synonymous_variant']
nonsynonymous_terms = [
    "start_lost",
    "missense_variant",
    "frameshift_variant",
    "stop_gained",
    "missense_variant&splice_region_variant",
    "inframe_deletion",
    "frameshift_variant&splice_region_variant",
    "inframe_insertion",
    "stop_gained&splice_region_variant",
    "stop_lost",
    "stop_gained&frameshift_variant",
    "frameshift_variant&stop_lost",
    "stop_gained&inframe_deletion",
    "frameshift_variant&start_lost",
]

parquet_dir = "/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data/parquets/"
AF_FILTERED = pd.read_parquet(f"{parquet_dir}AF_filtered_STG_final.parquet")
DP = pd.read_parquet(f"{parquet_dir}DP_filtered_STG_final.parquet")
metadata_filtered = pd.read_parquet(f"{parquet_dir}metadata_filtered_repeats_germline_freq.parquet")
GQ = pd.read_parquet(f"{parquet_dir}GQ_filtered_STG_final.parquet")
NGT_final = pd.read_parquet(f"{parquet_dir}NGT_filtered_STG_final.parquet")

metadata_filtered['age_at_death'] = metadata_filtered['age_at_death'].astype(str).str.replace(',', '.', regex=False)
metadata_filtered = metadata_filtered.loc[NGT_final.index]

# Load annotations
annotations = pd.read_csv('/results/rr/study/hg38s/study232-missionbio_TDP-C/outputs/vaf30/Annotations_vafhet30.csv', sep='\t', header=0) 
annotations['Protein_annotation'] = annotations['HGVSp'].str.split(".", expand=True)[2]
annotations = annotations[annotations['variant_id'].isin(NGT_final.columns)]
annotate_spliceai_effect(annotations)

genes = ["TARDBP", "TET2", "TMEM106B", "OPTN", "TBK1", "GRN", "UNC13A"]
annotations_df = []
NGT_list_filtered = []
NGT_patients_list = []
NGT_controls_list = []

ann_all_vars_list = []
ann_all_snv_list = []
ann_all_indels_list = []

ann_coding_list = []
ann_indels_list = []
ann_snv_list = []

ann_nonsyn_list = []
ann_syn_list = []
ann_missense_list = []
ann_nonsense_list = []
ann_frameshift_list = []
ann_splicesite_list = []
ann_startloss_list = []
ann_stoploss_list = []
cadd_above20_list = []
ann_LOF_list = []
ann_HC_LOF_list = []


var_groups_dict = {}
burden_dict = {}
var_groups_names = [
    "Coding_variants", "SNVs", "Indels", "Synonymous",
    "Non_synonymous", "Missense", "Nonsense",
    "Frameshift", "Splice site", "LOF", "HC_LOF", "CADD_atleast20",
]

for i, gene_name in enumerate(genes):
    ann = annotations[annotations['SYMBOL']==gene_name].copy()
    ann.index = ann['variant_id']
    ann["HC_LOF"] = ann.apply(categorize_hc_lof,  axis=1)
    ann = ann.set_index('variant_id')

    vars = ann.index.tolist()
    NGT = NGT_final.loc[:, vars].copy()
    meta = metadata_filtered.reindex(NGT.index).copy()

    NGT = NGT.assign(
        sample_identifier = meta['ID'],
        sample_id         = meta['sample_id'],
        sample_type       = meta['sample_type'],
        age_at_death      = meta['age_at_death'].to_numpy(),
        age_at_onset      = meta['age_at_onset'].to_numpy(),
        hemisphere        = meta['hemisphere'],
    )
    NGT_list_filtered.append(NGT)
    
    NGT_controls = NGT[NGT['sample_type'] == "C"]
    NGT_patients = NGT[NGT['sample_type'] == "P"]

    counts = pd.DataFrame({
        "cell_count_patients": NGT_patients.iloc[:, :-6].isin([1, 2]).sum(),
        "total_genotyped_cells_P": NGT_patients.iloc[:, :-6].isin([0, 1, 2]).sum()
    })
    counts["mutation_freq_P"] = counts["cell_count_patients"] / counts["total_genotyped_cells_P"] * 100
    ann = ann.merge(counts, left_index=True, right_index=True, how="inner")

    counts_C = pd.DataFrame({
        "cell_count_controls": NGT_controls.iloc[:, :-6].isin([1, 2]).sum(),
        "total_genotyped_cells_C": NGT_controls.iloc[:, :-6].isin([0, 1, 2]).sum()
    })
    counts_C["mutation_freq_C"] = counts_C["cell_count_controls"] / counts_C["total_genotyped_cells_C"] * 100
    ann = ann.merge(counts_C, left_index=True, right_index=True, how="inner")
    ann["total_cells"] = ann["cell_count_patients"] + ann["cell_count_controls"]
    ann["total_genotyped_cells"] = ann['total_genotyped_cells_P'] + ann['total_genotyped_cells_C']
    ann['total_freq'] = ann["total_cells"]/ann['total_genotyped_cells']*100

    annotations_df.append(ann)
    NGT_patients_list.append(NGT_patients)
    NGT_controls_list.append(NGT_controls)

    lengths_ref = ann['REF'].apply(len)
    length_alt = ann['ALT'].apply(len)
    indels = (lengths_ref[lengths_ref >1].index.to_list()) + (length_alt[length_alt >1].index.to_list()) # Count total number of indels
    indels = list(set(indels))
    ann_all_snv = ann.drop(index = indels) # To calculate total number of SNVs
    ann_all_indels = ann.loc[indels]

    ann_coding = ann[~ann['Consequence'].isin(non_coding_terms)]
    ann_indel = ann_coding[ann_coding['Consequence'].isin(indel_terms)] #Indels in coding regions 
    ann_snv = ann_coding[~ann_coding['Consequence'].isin(indel_terms)] # SNVs in coding regions
    
    ann_nonsyn = ann_coding[ann_coding['Consequence'].isin(nonsynonymous_terms)]
    ann_syn = ann_coding[ann_coding['Consequence'].isin(synonymous_terms)]
    ann_missense = ann_coding[ann_coding['Consequence'].isin(missense_terms)]
    ann_nonsense = ann_coding[ann_coding['Consequence'].isin(nonsense_terms)]
    ann_frameshift = ann_coding[ann_coding['Consequence'].isin(frameshift_terms)]
    ann_splicesite = ann_coding[ann_coding['Consequence'].isin(splice_site_terms)]
    ann_startloss = ann_coding[ann_coding['Consequence'].isin(["start_lost"])]
    ann_stoploss = ann_coding[ann_coding['Consequence'].isin(["stop_lost"])]
    ann_lof = ann_coding[ann_coding['Consequence'].isin(list(LOF))]
    ann_lof_hf = ann_lof[ann_lof['HC_LOF']=="Yes"]
    cadd_above20 = ann_snv[ann_snv['CADD_PHRED'] >= 20]

    ann_all_vars_list.append(ann)
    ann_all_snv_list.append(ann_all_snv)
    ann_all_indels_list.append(ann_all_indels)
    ann_snv_list.append(ann_snv)
    ann_indels_list.append(ann_indel)
    ann_coding_list.append(ann_coding)
    ann_nonsyn_list.append(ann_nonsyn)
    ann_syn_list.append(ann_syn)

    ann_missense_list.append(ann_missense)
    ann_nonsense_list.append(ann_nonsense)
    ann_frameshift_list.append(ann_frameshift)
    ann_splicesite_list.append(ann_splicesite)
    ann_startloss_list.append(ann_startloss)
    ann_stoploss_list.append(ann_stoploss)
    ann_LOF_list.append(ann_lof)
    ann_HC_LOF_list.append(ann_lof_hf)
    cadd_above20_list.append(cadd_above20)

    var_groups_dict[gene_name] = {
        group_name: group_list[i]
        for group_name, group_list in zip(
            var_groups_names,
            [
                ann_coding_list,
                ann_snv_list,
                ann_indels_list,
                ann_syn_list,
                ann_nonsyn_list,
                ann_missense_list,
                ann_nonsense_list,
                ann_frameshift_list,
                ann_splicesite_list,
                ann_LOF_list,
                ann_HC_LOF_list,
                cadd_above20_list

            ]
        )
    }

    df = NGT.copy()
    ids_selected = df['sample_id'].unique()
    list_vars = var_groups_dict[gene_name]
    

In [ ]:
for i,gene in enumerate(genes):
    print(gene)
    annotations_sel=ann_coding_list[i]
    vars = annotations_sel.index.to_list()
    print("Number of variants:", len(vars))
    print("Number of unique positions:", annotations_sel[annotations_sel['SYMBOL'] == gene]['POS'].nunique())
    cols_90_gt10 = DP[vars].columns[(DP[vars].gt(10).mean(axis=0) >= 0.90)]
    


In [ ]:
## Burden: Porportion of affected neuronal nuclei per sample harbouring a somatic variant
burden_dict = {}
for i, gene_name in enumerate(genes):
    df = NGT_list_filtered[i].copy()
    ids_selected = df['sample_id'].unique()
    snp_set = var_groups_dict[gene_name]['Coding_variants'].index.tolist()
    burden_dict[gene_name] = {}

    burdened_cells_per_sample = pd.DataFrame(index=ids_selected,columns=['burden', 'Group', 'n_count', 'n_variants', 'total_cells', 'median_DP'])
    
    for id in ids_selected:
        NGT_sample = df[df['sample_id'] == id]
        df_sample_selected_vars = NGT_sample[snp_set]
        
        x = DP.loc[df_sample_selected_vars.index, snp_set]
        parts = x.columns.to_series().str.split(":", expand=True)
        x.columns = parts.iloc[:, 1].to_numpy()
        x = x.loc[:, ~x.columns.duplicated()]
        sample_coverage = np.median(x.median(axis=0))

        df_varcounts = pd.DataFrame(df_sample_selected_vars.isin([1, 2]).sum(axis=1))
        mut_counts = df_varcounts[df_varcounts[0] > 0].shape[0]
        total_cells = df_sample_selected_vars.shape[0]
        n_variants = (NGT_sample.isin([1, 2])).any().sum()
        burden = np.round((mut_counts / total_cells), 6)
        burdened_cells_per_sample.loc[id, 'burden'] = burden
        burdened_cells_per_sample.loc[id, 'median_DP'] = sample_coverage
        burdened_cells_per_sample.loc[id, 'n_count'] = mut_counts
        burdened_cells_per_sample.loc[id, 'n_variants'] = n_variants
        burdened_cells_per_sample.loc[id, 'total_cells'] = total_cells
        
        if NGT_sample['sample_type'].unique().tolist() == ["P"]:
            burdened_cells_per_sample.loc[id, 'Group'] = "P"
        else:
            burdened_cells_per_sample.loc[id, 'Group'] = "C"
    
    burden_dict[gene_name] = burdened_cells_per_sample

In [ ]:
sample_metadata_path = '/results/rr/study/hg38s/study232-missionbio_TDP-C/data/metadata_hemisphere_PMI.csv'
sample_metadata = pd.read_csv(sample_metadata_path, sep=";")   
sample_metadata.set_index('Sample', inplace=True)
metadata = sample_metadata[sample_metadata['tissue'] == 'STG'].copy()
metadata["AGEATDEATH"] = (metadata["AGEATDEATH"].astype(str).str.replace(",", ".", regex=False))
metadata["AGEATDEATH"] = pd.to_numeric(metadata["AGEATDEATH"], errors="coerce")  # convert to float
metadata["AGEATONSET"] = (metadata["AGEATONSET"].astype(str).str.replace(",", ".", regex=False))
metadata["AGEATONSET"] = pd.to_numeric(metadata["AGEATONSET"], errors="coerce")  # convert to float

for sel_gene in genes:
    df_burden  = burden_dict[sel_gene]['Coding_variants'].copy()
    
    df_burden['Condition'] = df_burden.index.map(metadata['Group'])
    df_burden['Sex'] = df_burden.index.map(metadata['SEXatBirth'])
    df_burden['Age_at_Death'] = df_burden.index.map(metadata['AGEATDEATH'])
    df_burden['Recruitment_Site'] = df_burden.index.map(metadata['Recruitment_Site'])
    df_burden['Age_at_Death'] = pd.to_numeric(df_burden['Age_at_Death'], errors='coerce')
    df_burden['Pool'] = df_burden.index.map(metadata['pool'])
    df_burden['Hemisphere'] = df_burden.index.map(metadata['Hemisphere'])
    df_burden['AGEATONSET'] = df_burden.index.map(metadata['AGEATONSET'])
    df_burden.to_csv(f'/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data/Burden_files/gene/{sel_gene}_Coding_variants.tsv', sep="\t", index=True)


### TDP-43 domain specific burden

In [ ]:
## Select mutations
vars = var_groups_dict['TARDBP']['Coding_variants'].index
df = annotations_df[0].loc[vars]
df['codon'] = df['Protein_position'].copy()
df.loc['chr1:11018854:GCA/G',"codon"] = '175'
df["codon"] = pd.to_numeric(df["codon"])
len(vars)

In [ ]:
variants_NTD = df[df['codon']<= 76].index.to_list()
variants_RRM1 = df[(df['codon'] >= 104) & (df['codon'] <= 176)].index.to_list()
variants_RRM2 = df[(df['codon'] >= 192) & (df['codon'] <= 262)].index.to_list()
variants_CTD = df[(df['codon'] >= 274) & (df['codon'] <= 414)].index.to_list()

In [ ]:
print(len(variants_NTD))
print(len(variants_RRM1))
print(len(variants_RRM2))
print(len(variants_CTD))

In [ ]:
domain_names = ["NTD", "RRM1", "RRM2","CTD"]
domain_length = [198, 218, 165, 372]
domain_groups = [variants_NTD,  variants_RRM1, variants_RRM2, variants_CTD]

In [ ]:
## For different SNP groups
burden_list_domains = []
df = NGT_list_filtered[0]
ids_selected = df['sample_id'].unique()

for x in range(len(domain_groups)):
    snp_set = domain_groups[x]
    burdened_cells_per_domain = pd.DataFrame(index=ids_selected, columns=['ID', 'n_count', 'n_variants', 'total_cells', 'Condition', 'median_DP'])

    for id in ids_selected:
        df_sample = df[df['sample_id']==id]
        df_sample_selected_vars = df_sample[snp_set]
        df_per_cell = pd.DataFrame(df_sample_selected_vars.isin([1,2]).sum(axis=1))
        entry = df_per_cell[df_per_cell[0]>0].shape[0]
        n_cells = df_sample_selected_vars.shape[0]
        n_variants = (df_sample_selected_vars.isin([1, 2])).any().sum()
        
        x = DP.loc[df_sample_selected_vars.index, snp_set]
        parts = x.columns.to_series().str.split(":", expand=True)
        x.columns = parts.iloc[:, 1].to_numpy()
        x = x.loc[:, ~x.columns.duplicated()]
        sample_coverage_domain = np.median(x.median(axis=0))

        burdened_cells_per_domain.loc[id, 'ID'] = id
        burdened_cells_per_domain.loc[id, 'n_count'] = entry
        burdened_cells_per_domain.loc[id, 'total_cells'] = n_cells
        burdened_cells_per_domain.loc[id, 'n_variants'] = n_variants
        burdened_cells_per_domain.loc[id, 'median_DP'] = sample_coverage_domain
        if df_sample['sample_type'].unique().tolist() == ["P"]:
            burdened_cells_per_domain.loc[id, 'Condition'] = "P"
        else:
            burdened_cells_per_domain.loc[id, 'Condition'] = "C"
    burden_list_domains.append(burdened_cells_per_domain)


In [ ]:
burden_df = pd.concat([
        df.assign(domain = domain_names[i],domain_length = domain_length[i])
        for i, df in enumerate(burden_list_domains)],ignore_index=True)

In [ ]:
df_reg = burden_df.set_index('ID')
sample_metadata_path = '/results/rr/study/hg38s/study232-missionbio_TDP-C/data/metadata_hemisphere_PMI.csv'
sample_metadata = pd.read_csv(sample_metadata_path, sep=";")   
sample_metadata.set_index('Sample', inplace=True)
metadata = sample_metadata[sample_metadata['tissue'] == 'STG'].copy()
metadata["AGEATDEATH"] = (metadata["AGEATDEATH"].astype(str).str.replace(",", ".", regex=False))
metadata["AGEATDEATH"] = pd.to_numeric(metadata["AGEATDEATH"], errors="coerce")  # convert to float
#metadata["AGEATDEATH"] = metadata["AGEATDEATH"].fillna(metadata["AGEATDEATH"].mean())    

df_reg['Condition'] = df_reg.index.map(metadata['Group'])
df_reg['Sex'] = df_reg.index.map(metadata['SEXatBirth'])
df_reg['Age_at_Death'] = df_reg.index.map(metadata['AGEATDEATH'])
df_reg['Recruitment_Site'] = df_reg.index.map(metadata['Recruitment_Site'])
df_reg['Age_at_Death'] = pd.to_numeric(df_reg['Age_at_Death'], errors='coerce')
df_reg['Pool'] = df_reg.index.map(metadata['pool'])
df_reg['Hemisphere'] = df_reg.index.map(metadata['Hemisphere'])
df_reg.to_csv("/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data/Burden_files/tardp_domains/TARDBP_Coding_variants_perdomain.tsv", sep="\t", index=True)